# Session 3 Pythons basics in bioinformatics

Work from a Jupyter server started in Module02-02. This guided notebook uses
a real PhiX174 genome and a 100-read public FASTQ excerpt. Predict first,
execute second, and retain an independent check.


## Google Colab setup (optional)

Colab users should copy the repository folder to Google Drive, uncomment and
run the installation cell once per fresh runtime, and then uncomment the Drive
lines in the imports and paths cell below. Local Jupyter users can leave all
Colab-specific lines commented.


In [ ]:
# Google Colab only: uncomment and run once per fresh runtime.
# %pip install -q "biopython>=1.85,<2" "matplotlib>=3.9,<4" "numpy>=2,<3" "pandas>=2.2,<3"


In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from Bio import SeqIO

REPO_ROOT = None
# Google Colab only: uncomment these lines after copying the repository to Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# REPO_ROOT = Path("/content/drive/MyDrive/UIC_Bioinformatics_-_AI_for_Healthcare/")

# Local Jupyter default: discover the repository from the current directory.
if "REPO_ROOT" not in globals():
    cwd = Path.cwd().resolve()
    REPO_ROOT = next((
        candidate for candidate in (cwd, *cwd.parents)
        if (candidate / "Module02-02" / "data").is_dir()
    ), None)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repository not found. In Colab, mount Drive and set REPO_ROOT above."
    )

module_dir = Path(REPO_ROOT) / "Module02-02"
data_dir = module_dir / "data" / "teaching"
fasta_path = data_dir / "phix174.fasta"
fastq_path = data_dir / "phix_reads_100.fastq"
orchid_fasta_path = data_dir / "orchid" / "ls_orchid.fasta"
orchid_genbank_path = data_dir / "orchid" / "ls_orchid.gbk"
required_paths = [fasta_path, fastq_path, orchid_fasta_path, orchid_genbank_path]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Missing required data files: {missing_paths}")
print("Module directory:", module_dir.resolve())


In [ ]:
print("FASTA preview:")
print("\n".join(fasta_path.read_text().splitlines()[:3]))
print("\nFASTQ preview:")
print("\n".join(fastq_path.read_text().splitlines()[:8]))


## Python mental model

A path names a file, a string stores text, a dictionary maps identifiers to
sequences, and a function defines a reusable contract. A loop changes state
one record at a time. State explicitly what each object represents.


In [ ]:
example = "ACGTNN"
observations = {
    "length": len(example),
    "starts_with_A": example.startswith("A"),
    "symbols": sorted(set(example)),
}
for key, value in observations.items():
    print(key, type(value).__name__, value)


## Manual FASTA parser

FASTA sequences may span lines. Track the current identifier, accumulate
sequence "fragments", and join only when a record ends. Reject sequence text that
appears before the first header.


In [ ]:
def read_fasta(path):
    records = {}
    current_id = None
    fragments = []

    for line_number, raw_line in enumerate(Path(path).read_text().splitlines(), 1):
        line = raw_line.strip()
        # Through vibe coding / AI agents finish the loop to read the FASTA file and populate the records dictionary.
        # Check for duplicate identifiers and consider raising an error.
    if current_id is not None:
        records[current_id] = "".join(fragments).upper()
    return records


In [ ]:
# Test the read_fasta function with the phix174.fasta file and this custom assertions
manual_fasta = read_fasta(fasta_path)
assert len(manual_fasta) == 1
phix_id, phix_sequence = next(iter(manual_fasta.items()))
assert len(phix_sequence) == 5386
print(phix_id, len(phix_sequence), sorted(set(phix_sequence)))


## Even mall functions need clear structuring

Decide case handling, allowed symbols, denominators, and empty-input
behavior before coding. Assertions allow to document specific cases of use as executable examples.


In [ ]:
def count_bases(sequence : str, alphabet="ACGTN"):
    """This function counts the occurrences of each base in a given sequence.
    """
    counts = Counter(sequence.upper())
    return {base: counts.get(base, 0) for base in alphabet}

base_counts = count_bases(phix_sequence)
base_counts


In [ ]:
# Using as an inspiration count_bases, implement a function that counts the gc_fraction of a sequence.

In [ ]:
expected_gc = 0.44764203490531007
observed_gc = gc_fraction(phix_sequence)
assert abs(observed_gc - expected_gc) < 1e-12
assert gc_fraction("gcNN") == 1.0
print(f"PhiX174 GC fraction: {observed_gc:.4f}")


In [ ]:
# Using as an inspiration count_bases and gc_fraction, implement a function that counts the number of occurrences of each k-mer in a given sequence. The function should take two arguments: the sequence and the value of k. It should return a dictionary with k-mers as keys and their counts as values.

## NumPy: the mathematical module for Python

Arrays have shape and dtype. They are helpful when many values share one
numerical meaning; they do not replace identifiers or biological metadata.

It is also widely used in any function or script which requires intensive mathematical application.


In [ ]:
# Useful to generate a numeric view of the base counts and to latter use it to generate plots.
bases = np.array(list("ACGT"))
counts = np.array([base_counts[base] for base in bases])
print("shape:", counts.shape, "dtype:", counts.dtype)
print(counts)


## pandas: the module involved in Big Data operations

In this example we are going to show a labelled table.
A DataFrame makes columns and rows explicit. Choose a schema before filling
it, and keep biological units in the column names or documentation.


In [ ]:
sequence_summary = pd.DataFrame([{
    "record_id": phix_id,
    "length_bp": len(phix_sequence),
    "gc_fraction": observed_gc,
    # This line uses dictionary logic to create new columns for each element in base = ("A", "C", "G", "T")
    **{f"count_{base}": base_counts[base] for base in "ACGT"},
}])
sequence_summary


## Matplotlib: the module adapted for python plotting (pyplot)

This module is the staple in python plotting, almost as much as ggplot2 is for R. Here we'll generate a quick barplot for our basecount.

In [ ]:
# Let's plot them with pyplot (from matplotlib).
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(bases, counts, color="#00798c")
ax.set(xlabel="Base", ylabel="Count", title="PhiX174 base composition")
ax.grid(axis="y", alpha=0.2)
plt.show()


In [ ]:
# Can you generate a similar plot but with different colors for each base?

## FASTQ adds per-base quality

FASTQ files are caracterized by four lines: header, sequence, separator, and ASCII symbols for its
sequence qualities. 

Sequence and quality lengths must agree. Pairing and wrapped
records require more than a line-count shortcut.


In [ ]:
def read_fastq_four_line(path):
    lines = Path(path).read_text().splitlines()
    if len(lines) % 4:
        raise ValueError("FASTQ line count is not divisible by four")
    records = []
    # Now vibe code use an AI agents finish the loop to read the FASTQ file and populate the records dictionary.
    return records


In [ ]:
# Use this assertions to test it
manual_fastq = read_fastq_four_line(fastq_path)
assert len(manual_fastq) == 100
read_lengths = np.array([len(sequence) for _, sequence, _ in manual_fastq])
phred_values = np.array([
    ord(character) - 33
    for _, _, quality in manual_fastq
    for character in quality
])
print("reads:", len(manual_fastq))
print("read lengths:", read_lengths.min(), read_lengths.max())
print("Phred range:", phred_values.min(), phred_values.max())


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
# Now plot using plt.hist (histogram) to visualize the distribution of Phred quality scores. Tip, there are up to 46 unique scores.
plt.show()


## Compare with Biopython

Biopython is the alma mater of python usage in bioinformatic contexts. Its modules and functions are extremly tailored to work with bioinformatic files and records.

SeqIO yields SeqRecord objects with identifiers, sequences, descriptions,
and annotations. 

Check whether it agrees with our manual parser, however keep in mind that it is not
not proof that every biological assumption is correct.


In [ ]:
bio_fasta = list(SeqIO.parse(fasta_path, "fasta"))
bio_fastq = list(SeqIO.parse(fastq_path, "fastq"))
# Design 4 assertions: 1) That the sequence is exactly the same with phix_sequence, 
# 2) that the length of the bio_fasta and manual_fasta is 1, 
# 3) that the length of the bio_fastq and manual_fastq is 100, 
# 4) that the id of the first record in bio_fastq is the same as the id of the first record in manual_fastq.
print("Manual and Biopython record models agree on tested properties.")


## One sequence, two representations:  FASTA and GenBank

FASTA is a compact sequence exchange format; GenBank can retain descriptions,
references, annotations, and sequence features. Compare a bounded record
rather than printing all records and metadata.



In [ ]:
orchid_fasta = list(SeqIO.parse(orchid_fasta_path, "fasta"))
orchid_genbank = list(SeqIO.parse(orchid_genbank_path, "genbank"))
assert len(orchid_fasta) == len(orchid_genbank) == 94

# The historical FASTA uses pipe-delimited identifiers while GenBank uses
# the accession.version directly. Normalize only the join key and retain
# both original identifiers as evidence.
fasta_by_id = {
    (record.id.split("|")[3] if "|" in record.id else record.id): record
    for record in orchid_fasta
}
genbank_by_id = {record.id: record for record in orchid_genbank}
shared_id = sorted(fasta_by_id)[0]
assert str(fasta_by_id[shared_id].seq).upper() == str(genbank_by_id[shared_id].seq).upper()

orchid_comparison = pd.DataFrame([{
    "record_id": shared_id,
    "length_bp": len(genbank_by_id[shared_id]),
    "fasta_description": fasta_by_id[shared_id].description,
    "genbank_feature_count": len(genbank_by_id[shared_id].features),
    "genbank_annotation_keys": ", ".join(sorted(genbank_by_id[shared_id].annotations)),
}])
orchid_comparison


## AI review checkpoint

**Izumi**, a PhD student of the group ask you for help. She has multiple deadlines and cannot work in this specific part of her PhD project involved with orchid plants. Can you help her finish parsing and plotting her data?

Ask an AI assistant to complete `build_orchid_summary()` using the contract
below. The function must combine sequence evidence from FASTA with organism
and feature metadata from GenBank.

Before accepting the candidate:

1. Inspect how identifiers are normalized.
2. Test duplicate and missing identifiers.
3. Confirm that all 94 sequences agree across formats.
4. Check how ambiguous bases affect the GC denominator.
5. Plot the verified table and explain one pattern and one limitation.


In [ ]:
def build_orchid_summary(fasta_records, genbank_records):
    """
    Return one DataFrame row per orchid accession.

    Required columns:
      record_id
      organism
      genus
      fasta_length_bp
      genbank_length_bp
      sequences_match
      gc_fraction_valid
      non_acgt_bp
      non_acgt_fraction
      genbank_feature_count

    Requirements:
      - Normalize genbank identifiers with the FASTA identifiers.
      - Reject duplicate identifiers.
      - Reject unequal FASTA and GenBank identifier sets.
      - Calculate GC over valid A/C/G/T bases. Use previous functions to do so.
      - Do NOT modify the input records.
    """
    rows = [] # We will append each row as a dictionary to this list. 

    def normalize_id(record_id):
        """Convert a historical pipe-delimited ID to accession.version."""
        parts = record_id.split("|")

        # Example: gi|2765658|emb|Z78533.1|CIZ78533
        if len(parts) >= 4 and parts[3]:
            return parts[3]

        return record_id
    
    # Vibe code a index_unique internal function and then complete the rest of the build_orchid_summary.
    return pd.DataFrame(rows).sort_values("record_id").reset_index(drop=True)

In [ ]:
# Verification cell to test whether your implementation works. Do not modify this cell neither pass it through your AI agent.
orchid_summary = build_orchid_summary(orchid_fasta, orchid_genbank)

assert len(orchid_summary) == 94
assert orchid_summary["record_id"].is_unique
assert orchid_summary["sequences_match"].all()
assert (
    orchid_summary["fasta_length_bp"]
    == orchid_summary["genbank_length_bp"]
).all()

expected_genus_counts = {
    "Paphiopedilum": 61,
    "Cypripedium": 17,
    "Phragmipedium": 15,
    "Mexipedium": 1,
}

assert orchid_summary["genus"].value_counts().to_dict() == expected_genus_counts
assert orchid_summary["non_acgt_bp"].sum() == 541
assert (orchid_summary["genbank_feature_count"] == 5).all()
assert orchid_summary["gc_fraction_valid"].between(0, 1).all()

orchid_summary.head()

## AI-use record and exit checkpoint

Use the following template.


| Field | Record |
|---|---|
| Session and task | |
| Prompt | |
| Candidate received | |
| Assumptions detected | |
| Tests and expected results | |
| Observed evidence | |
| Correction, if any | |
| Decision: accept, correct, or reject | |
| Remaining limitation | |

Attach the tested command or function when it is needed to reproduce the
decision. Do not paste an entire conversation when a bounded prompt, relevant
candidate, and evidence are sufficient.